# Anthropic Quick Reference

This notebook is a compact companion to the scripts in `02_anthropic/scripts`.
It focuses on the most common actions you will repeat when working with the Anthropic Python SDK:

- create a client
- generate plain text
- parse structured outputs
- stream text incrementally
- create embeddings (via VoyageAI)
- send multi-turn message inputs

In [ ]:
import math
import sys
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "02_anthropic" / "scripts").exists():
            return candidate
    raise RuntimeError("Could not find the project root from the current notebook path")


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / "02_anthropic" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))


## Setup

The helper module in `02_anthropic/scripts/anthropic_client.py` keeps these examples short and close to the rest of the labs.

In [ ]:
from anthropic_client import (
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_TEXT_MODEL,
    generate_embedding,
    generate_structured,
    generate_text,
    generate_text_from_messages,
    get_client,
    stream_response,
)

load_dotenv(PROJECT_ROOT / ".env")
client = get_client()
DEFAULT_TEXT_MODEL

## 1. Plain Text Generation

Use `messages.create(...)` when you want normal text output.

In [ ]:
answer = generate_text(
    "Explain what an API is in 3 short bullet points.",
    system="You are a concise Python tutor.",
    client=client,
)

print(answer)

## 2. Structured Outputs With Pydantic

Use tool use under the hood to enforce the JSON schema from a Pydantic model.

In [ ]:
class LessonPlan(BaseModel):
    title: str
    difficulty: str
    estimated_minutes: int = Field(ge=1)
    key_points: list[str]


lesson_plan = generate_structured(
    "Create a short beginner lesson plan about Python lists.",
    output_model=LessonPlan,
    system="Return a concise educational plan.",
    client=client,
)

lesson_plan

## 3. Streaming

Streaming is useful when you want to display text progressively in a UI or CLI.

In [ ]:
for delta in stream_response(
    "Write a short explanation of streaming responses in 3 sentences.",
    client=client,
):
    print(delta, end="", flush=True)

print()

## 4. Embeddings (via VoyageAI)

Anthropic does not expose a native embedding API. VoyageAI is the recommended
partner provider for embeddings used alongside Claude.

Embeddings turn text into vectors so you can compare semantic similarity.

In [ ]:
def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b, strict=True))
    norm_a = math.sqrt(sum(v * v for v in vector_a))
    norm_b = math.sqrt(sum(v * v for v in vector_b))
    return dot_product / (norm_a * norm_b)


vector_a = generate_embedding("How do I sort a Python list?")
vector_b = generate_embedding(
    "What is the best way to order items in a Python list?"
)

print(f"Model: {DEFAULT_EMBEDDING_MODEL}")
print(f"Cosine similarity: {cosine_similarity(vector_a, vector_b):.4f}")

## 5. Multi-turn Messages

When the input is more complex than one string, send a full message list
with alternating user/assistant turns.

In [ ]:
messages = [
    {"role": "user", "content": "What is a Python tuple?"},
    {"role": "assistant", "content": "A tuple is an immutable, ordered sequence."},
    {
        "role": "user",
        "content": "How does it differ from a list? Answer in one sentence."
    },
]

generate_text_from_messages(
    messages,
    system="You are a concise coding assistant.",
    client=client,
)

## Where To Go Next

Use this notebook for quick iteration, then switch to the topic notebooks for deeper dives:

- `00_notebook_basic_API.ipynb` — SDK setup and basic request patterns
- `02_prompting.ipynb` — prompt engineering and evaluation
- `03_tools.ipynb` — tool use and agentic loops
- `03_tool_streaming.ipynb` — streaming with tools
- `04_embeddings.ipynb` / `04_hybrid.ipynb` — retrieval and RAG
- `05_thinking.ipynb` / `05_caching.ipynb` — advanced features